In [6]:
%load_ext autoreload
%autoreload 2

from datetime import datetime, date
from okx.store import OrderbookStore, populate, FEATURES
import polars as pl

# Initialize the store
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite"
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
# Populate BTC-USD-SWAP data for a few days
populate(
    store,
    inst_type='SWAP',
    inst_family='BTC-USD',
    start=datetime(2025, 8, 1),
    end=datetime(2025, 10, 1),
    verbose=True,
    max_workers=4
)

Fetching BTC-USD/SWAP: 2 missing dates (from 61 requested)


  2025-08-07: No data returned from API
  2025-08-09: No data returned from API
✓ Stored 0/2 days as raw orderbook data
  Failed dates: [datetime.date(2025, 8, 7), datetime.date(2025, 8, 9)]


In [ ]:
# Store deletion example
# Example: Delete raw orderbook data for a specific family/type/date

# Let's delete one day's data for BTC-USD FUTURES (e.g. August 15, 2025)
delete_date = date(2025, 8, 15)

# store.delete_raw('BTC-USD', 'FUTURES', delete_date) (uncomment to delete)

# You can check if it's deleted from the manifest:
have_after = store.manifest.have('BTC-USD', 'FUTURES', delete_date, 'raw')
print(f"Raw for {delete_date} present after delete? {have_after}")


In [ ]:
lf = store.get(
    inst_type='FUTURES',
    inst_family='BTC-USD',
    start=datetime(2024, 6, 1),
    end=datetime(2024, 6, 2),
    depth=0,
    features=['mid', 'trim']
    
)

# Collect and display
df = lf.collect()

In [4]:
print(f"Shape: {df.shape}")
df.head()

Shape: (10305142, 33)


timeMs,exchTimeMs,symbol,bid_1_px,bid_1_qty,bid_1_ordCnt,ask_1_px,ask_1_qty,ask_1_ordCnt,bid_2_px,bid_2_qty,bid_2_ordCnt,ask_2_px,ask_2_qty,ask_2_ordCnt,bid_3_px,bid_3_qty,bid_3_ordCnt,ask_3_px,ask_3_qty,ask_3_ordCnt,bid_4_px,bid_4_qty,bid_4_ordCnt,ask_4_px,ask_4_qty,ask_4_ordCnt,bid_5_px,bid_5_qty,bid_5_ordCnt,ask_5_px,ask_5_qty,ask_5_ordCnt
i64,i64,str,f64,f64,i32,f64,f64,i32,f64,f64,i32,f64,f64,i32,f64,f64,i32,f64,f64,i32,f64,f64,i32,f64,f64,i32,f64,f64,i32,f64,f64,i32
1717200000064,1717200000060,"""BTC-USD-240607.OK""",67666.6,32.0,1,67681.8,7.0,1,67656.4,12.0,1,67681.9,12.0,1,67654.2,2.0,1,67684.1,2.0,1,67647.8,12.0,1,67689.5,7.0,1,67647.5,2.0,1,67689.6,12.0,1
1717200000074,1717200000070,"""BTC-USD-240607.OK""",67666.6,32.0,1,67667.0,32.0,1,67656.4,12.0,1,67681.8,7.0,1,67654.2,2.0,1,67681.9,12.0,1,67647.8,12.0,1,67684.1,2.0,1,67647.5,2.0,1,67689.5,7.0,1
1717200000533,1717200000530,"""BTC-USD-240607.OK""",67666.6,32.0,1,67681.8,7.0,1,67656.4,12.0,1,67681.9,12.0,1,67654.2,2.0,1,67684.1,2.0,1,67647.8,12.0,1,67689.5,7.0,1,67647.5,2.0,1,67689.6,12.0,1
1717200000543,1717200000540,"""BTC-USD-240607.OK""",67666.6,32.0,1,67681.8,7.0,1,67656.4,12.0,1,67681.9,12.0,1,67654.2,2.0,1,67684.1,2.0,1,67647.8,12.0,1,67689.5,7.0,1,67647.5,2.0,1,67689.6,12.0,1
1717200000714,1717200000710,"""BTC-USD-240607.OK""",67666.6,32.0,1,67667.7,32.0,1,67656.4,12.0,1,67681.8,7.0,1,67654.2,2.0,1,67681.9,12.0,1,67647.8,12.0,1,67684.1,2.0,1,67647.5,2.0,1,67689.5,7.0,1


In [ ]:
print(df['symbol'].unique())

shape: (652,)
Series: 'symbol' [str]
[
	"BTC-USD-250919-116000-C.OK"
	"BTC-USD-250903-112000-P.OK"
	"BTC-USD-250926-116000-C.OK"
	"BTC-USD-251031-140000-C.OK"
	"BTC-USD-260327-40000-C.OK"
	…
	"BTC-USD-250912-115000-P.OK"
	"BTC-USD-250906-120000-P.OK"
	"BTC-USD-251128-114000-C.OK"
	"BTC-USD-250912-112000-P.OK"
	"BTC-USD-250903-107000-C.OK"
]


In [15]:
from utils import parse_option_name

# Since df is a Polars DataFrame (not a lazyframe at this point, since .collect() was called earlier), proceed accordingly.

# Analyze null count and percentage of each column
num_rows = df.height
nulls = df.null_count()
# Polars DataFrame returned by null_count() is a DataFrame, not a dict
# Convert to dict: use to_dict(as_series=False)[column_name][0]
nulls_dict = {col: nulls.select(col).to_series()[0] for col in df.columns}
nulls_pct = {col: (count / num_rows) * 100 for col, count in nulls_dict.items()}

# Display as a sorted table for readability, with both count and percentage
print("Null count and percentage per column:")
print(f"{'Column':<20} {'Nulls':>10} {'Pct':>8}")
for col in sorted(df.columns):
    count = nulls_dict[col]
    pct = nulls_pct[col]
    print(f"{col:<20} {count:10d} {pct:7.2f}%")

# Compute all unique option combos using parse_option_name
unique_symbols = df['symbol'].unique()
combos = set()
for sym in unique_symbols:
    # skip null/None values if present
    if sym is None:
        continue
    try:
        parsed = parse_option_name(sym)
        combo = (parsed[0], parsed[1], parsed[2])  # e.g. ('BTC-USD', expiry_datetime, strike)
        combos.add(combo)
    except Exception as e:
        print(f"Could not parse symbol '{sym}': {e}")

print("\nAll unique option combos (underlying, expiry, strike):")
for combo in sorted(combos, key=lambda x: (x[0], x[1], x[2])):
    print(combo)



Null count and percentage per column:
Column                    Nulls      Pct
ask_1_ordCnt             210635    0.72%
ask_1_px                 210635    0.72%
ask_1_qty                210635    0.72%
ask_2_ordCnt             231459    0.79%
ask_2_px                 231459    0.79%
ask_2_qty                231459    0.79%
ask_3_ordCnt             437901    1.50%
ask_3_px                 437901    1.50%
ask_3_qty                437901    1.50%
ask_4_ordCnt             852585    2.92%
ask_4_px                 852585    2.92%
ask_4_qty                852585    2.92%
ask_5_ordCnt            2393340    8.19%
ask_5_px                2393340    8.19%
ask_5_qty               2393340    8.19%
bid_1_ordCnt              30601    0.10%
bid_1_px                  30601    0.10%
bid_1_qty                 30601    0.10%
bid_2_ordCnt              64880    0.22%
bid_2_px                  64880    0.22%
bid_2_qty                 64880    0.22%
bid_3_ordCnt             182854    0.63%
bid_3_px           

In [16]:
print(len(df))

29218687
